# RFM Customer Segmentation
**Dataset:** UCI Online Retail (transaksi e-commerce UK, 2010–2011)

**Tujuan bisnis:** Identifikasi segmen pelanggan berdasarkan perilaku transaksi untuk mendukung strategi retensi dan marketing yang lebih tepat sasaran.

---
## Tahap 1 — Load Dataset

In [9]:
import os
print(os.listdir())

['.ipynb_checkpoints', 'Online Retail.csv', 'rfm_01_load_and_clean.ipynb', 'rfm_02_score_and_segment.ipynb', 'rfm_03_visualization.ipynb']


In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Styling konsisten untuk semua chart
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('Online Retail.csv')

print(f'Shape awal: {df.shape}')
df.head()

Shape awal: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [11]:
# Eksplorasi awal: tipe data dan ringkasan
df.info()
print('\nJumlah missing value per kolom:')
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB

Jumlah missing value per kolom:
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64


In [12]:
# Cek keseluruhan
print(df.shape)
print(df.dtypes)
print('\n--- Missing values ---')
print(df.isnull().sum())

# Cek kolom StockCode 
print('\nContoh StockCode unik (10 pertama):')
print(df['StockCode'].unique()[:10])
print('Jumlah StockCode unik:', df['StockCode'].nunique())

# Cek kolom Description 
print('\nContoh Description kosong:')
print(df[df['Description'].isnull()].head(3))

# Cek kolom Country
print('\nDistribusi Country (top 10):')
print(df['Country'].value_counts().head(10))

# Cek InvoiceDate 
print('\nRentang tanggal:')
print('Dari:', df['InvoiceDate'].min())
print('Sampai:', df['InvoiceDate'].max())

# Cek duplikat
print('\nJumlah baris duplikat:', df.duplicated().sum())

(1067371, 8)
Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object

--- Missing values ---
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

Contoh StockCode unik (10 pertama):
['85048' '79323P' '79323W' '22041' '21232' '22064' '21871' '21523' '22350'
 '22349']
Jumlah StockCode unik: 5305

Contoh Description kosong:
     Invoice StockCode Description  Quantity          InvoiceDate  Price  \
470   489521     21646         NaN       -50  2009-12-01 11:44:00    0.0   
3114  489655     20683         NaN       -44  2009-12-01 17:26:00    0.0   
3161  489659     21350         NaN       230  2009-12-01 17:39:00    0.0   

      Customer ID         Country  
470           NaN  United Kingdom  
3114          NaN  Un

In [13]:
# Cek StockCode yang bukan kode produk normal
kode_aneh = df[~df['StockCode'].str.match(r'^\d{5}', na=False)]['StockCode'].unique()
print('StockCode tidak standar:', kode_aneh[:20])

StockCode tidak standar: ['POST' 'D' 'DCGS0058' 'DCGS0068' 'DOT' 'M' 'DCGS0004' 'DCGS0076' 'C2'
 'BANK CHARGES' 'DCGS0003' 'TEST001' 'gift_0001_80' 'DCGS0072'
 'gift_0001_20' 'DCGS0044' 'TEST002' 'gift_0001_10' 'gift_0001_50'
 'DCGS0066N']


In [14]:
# Lihat contoh transaksi DCGS dan gift
print(df[df['StockCode'].str.startswith('DCGS')][['StockCode','Description','Price','Quantity']].head(5))
print()
print(df[df['StockCode'].str.startswith('gift')][['StockCode','Description','Price','Quantity']].head(5))


     StockCode                   Description  Price  Quantity
2377  DCGS0058              MISO PRETTY  GUM   0.83         1
2378  DCGS0068             DOGS NIGHT COLLAR   8.65         1
8371  DCGS0004    HAYNES CAMPER SHOULDER BAG  17.35         1
8372  DCGS0058              MISO PRETTY  GUM   0.83         1
8373  DCGS0076  SUNJAR LED NIGHT NIGHT LIGHT  16.48         1

          StockCode                         Description  Price  Quantity
30620  gift_0001_80                                 NaN   0.00         2
31079  gift_0001_80  Dotcomgiftshop Gift Voucher £80.00  69.56         1
32048  gift_0001_20  Dotcomgiftshop Gift Voucher £20.00  17.39         2
40904  gift_0001_10  Dotcomgiftshop Gift Voucher £10.00   8.69         1
40905  gift_0001_20  Dotcomgiftshop Gift Voucher £20.00  17.39         1


In [15]:
# Cek nilai negatif di Quantity dan Price
print('Quantity negatif:', (df['Quantity'] < 0).sum())
print('Price negatif:', (df['Price'] < 0).sum())
print('Price = 0:', (df['Price'] == 0).sum())

# Cek prefix di Invoice
print('\nContoh Invoice dengan prefix C:')
print(df[df['Invoice'].astype(str).str.startswith('C')]['Invoice'].head(10).values)

print('\nJumlah Invoice berawalan C:', df['Invoice'].astype(str).str.startswith('C').sum())

Quantity negatif: 22950
Price negatif: 5
Price = 0: 6202

Contoh Invoice dengan prefix C:
['C489449' 'C489449' 'C489449' 'C489449' 'C489449' 'C489449' 'C489449'
 'C489449' 'C489449' 'C489459']

Jumlah Invoice berawalan C: 19494


## Tahap 2 — Data Cleaning

Masalah yang ditemukan (dari hasil eksplorasi):

Customer ID ada yang kosong → 243.007 baris tidak bisa diatribusikan ke pelanggan
Invoice yang diawali 'C' → 19.494 transaksi cancelled
Quantity negatif → 22.950 baris transaksi retur
Price negatif → 5 baris, dan Price = 0 → 6.202 baris (error input & produk gratis)
StockCode non-produk → ditemukan kode seperti POST, DOT, BANK CHARGES, TEST001, gift_0001_xx yang bukan transaksi produk nyata
Duplikat → 34.335 baris identik

Keputusan bisnis:

Hanya gunakan transaksi dari pelanggan yang teridentifikasi (Customer ID tidak kosong)
Hanya transaksi pembelian valid — bukan retur, bukan cancellation
Hanya produk nyata — biaya pengiriman, diskon, voucher, dan data test dikeluarkan karena tidak merepresentasikan perilaku pembelian pelanggan



In [18]:
df_clean = df.copy()

# Hapus baris tanpa Customer ID 
before = len(df_clean)
df_clean = df_clean.dropna(subset=['Customer ID'])
print(f'Hapus tanpa Customer ID : -{before - len(df_clean):,} baris')

# Hapus transaksi cancelled (Invoice prefix 'C')
before = len(df_clean)
df_clean = df_clean[~df_clean['Invoice'].astype(str).str.startswith('C')]
print(f'Hapus transaksi cancelled: -{before - len(df_clean):,} baris')

# Hapus Quantity <= 0 (retur & error) 
before = len(df_clean)
df_clean = df_clean[df_clean['Quantity'] > 0]
print(f'Hapus Quantity <= 0      : -{before - len(df_clean):,} baris')

# Hapus Price <= 0 (gratis, error, & manual entry) 
before = len(df_clean)
df_clean = df_clean[df_clean['Price'] > 0]
print(f'Hapus Price <= 0         : -{before - len(df_clean):,} baris')

# Hapus StockCode non-produk 
kode_hapus = ['POST', 'D', 'DOT', 'M', 'BANK CHARGES', 'C2', 'TEST001', 'TEST002']
before = len(df_clean)
df_clean = df_clean[~df_clean['StockCode'].isin(kode_hapus)]
df_clean = df_clean[~df_clean['StockCode'].str.startswith('gift_', na=False)]
print(f'Hapus StockCode non-produk: -{before - len(df_clean):,} baris')

# Hapus duplikat
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f'Hapus duplikat           : -{before - len(df_clean):,} baris')

# Fix tipe data 
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['Customer ID'] = df_clean['Customer ID'].astype(int)

# Tambah kolom TotalPrice
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['Price']

# Ringkasan hasil cleaning
print(f'\n{"="*45}')
print(f'Shape awal            : {df.shape[0]:,} baris, {df.shape[1]} kolom')
print(f'Shape setelah cleaning: {df_clean.shape[0]:,} baris, {df_clean.shape[1]} kolom')
print(f'Baris terhapus        : {df.shape[0] - df_clean.shape[0]:,} baris')
print(f'{"="*45}')
print(f'Pelanggan unik        : {df_clean["Customer ID"].nunique():,}')
print(f'Produk unik           : {df_clean["StockCode"].nunique():,}')
print(f'Negara                : {df_clean["Country"].nunique()}')
print(f'Rentang tanggal       : {df_clean["InvoiceDate"].min().date()} s/d {df_clean["InvoiceDate"].max().date()}')
print(f'Total revenue         : £{df_clean["TotalPrice"].sum():,.0f}')
print(f'\nMissing values tersisa:\n{df_clean.isnull().sum()}')

# Simpan
import os
os.makedirs('../data', exist_ok=True)
df_clean.to_csv('../data/data_clean.csv', index=False)
print('Data bersih disimpan ke data/data_clean.csv')

Hapus tanpa Customer ID : -243,007 baris
Hapus transaksi cancelled: -18,744 baris
Hapus Quantity <= 0      : -0 baris
Hapus Price <= 0         : -71 baris
Hapus StockCode non-produk: -2,863 baris
Hapus duplikat           : -26,055 baris

Shape awal            : 1,067,371 baris, 8 kolom
Shape setelah cleaning: 776,631 baris, 9 kolom
Baris terhapus        : 290,740 baris
Pelanggan unik        : 5,861
Produk unik           : 4,623
Negara                : 41
Rentang tanggal       : 2009-12-01 s/d 2011-12-09
Total revenue         : £17,072,852

Missing values tersisa:
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
TotalPrice     0
dtype: int64
Data bersih disimpan ke data/data_clean.csv


In [ ]:
df_clean.head()